In [ ]:
import cv2
import pyttsx3
import speech_recognition as sr
from ultralytics import YOLO

# ==============================
# 1. إعداد المحرك الصوتي
# ==============================
engine = pyttsx3.init()
engine.setProperty("rate", 160)  # سرعة الكلام
engine.setProperty("volume", 1)  # مستوى الصوت

def speak(text):
    print(f"🤖: {text}")
    engine.say(text)
    engine.runAndWait()

# ==============================
# 2. الاستماع والتعرف على الكلام
# ==============================
def listen_and_recognize():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("🎤: استمع...")
        recognizer.adjust_for_ambient_noise(source)
        try:
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
            text = recognizer.recognize_google(audio, language="ar-SA")
            print(f"👂: {text}")
            return text
        except sr.WaitTimeoutError:
            print("⚠: لم يتم سماع أي صوت")
            return ""
        except sr.UnknownValueError:
            print("⚠: مش قادر افهم الكلام")
            return ""
        except sr.RequestError:
            print("⚠: مشكلة في خدمة التعرف")
            return ""

# ==============================
# 3. فتح الكاميرا
# ==============================
def open_camera():
    """يحاول يفتح الكاميرا على أكثر من index"""
    for i in range(3):  # جرب 0 و 1 و 2
        cap = cv2.VideoCapture(i)
        if cap.isOpened():
            print(f"✅ الكاميرا اتفتحت على index {i}")
            return cap
        cap.release()
    return None

# ==============================
# 4. أوامر الروبوت
# ==============================
def understand_command(text):
    if "كوب" in text or "مشروب" in text:
        return "cup"
    elif "تليفون" in text or "موبايل" in text:
        return "phone"
    elif "اشرب" in text:
        return "drink"
    elif "رد" in text:
        return "answer"
    elif "اقفل" in text:
        return "hangup"
    else:
        return "unknown"

def control_robot_arm(action, target_coords=None, face_coords=None):
    if action == "cup":
        speak("🤖: همسك الكوب 🥤")
    elif action == "phone":
        speak("🤖: همسك التليفون 📱")
    elif action == "drink" and face_coords:
        speak("🤖: هشرب بالنيابة عنك 🥤")
    elif action == "answer":
        speak("🤖: هرد على التليفون 📞")
    elif action == "hangup":
        speak("🤖: هقفل التليفون ❌")
    else:
        speak("⚠ مش قادر أنفذ الأمر")

# ==============================
# 5. البرنامج الرئيسي
# ==============================
if __name__ == "__main__":
    speak("مرحبًا، أنا جاهز للأوامر!")

    # تحميل موديل YOLO
    model = YOLO("yolov8n.pt")

    # تحميل كاشف الوجوه
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    # فتح الكاميرا
    cap = open_camera()
    if cap is None:
        speak("⚠ مش لاقي أي كاميرا متصلة!")
        raise SystemExit

    while True:
        # الاستماع للأوامر
        text = listen_and_recognize()

        # أوامر الخروج
        if "شكرا" in text or "خروج" in text or "انهاء" in text:
            speak("العفو، إلى اللقاء 👋")
            break

        # تنظيف الكلام
        if text.startswith("انا عاوزه"):
            text = text.replace("انا عاوزه", "").strip()
        elif text.startswith("انا محتاجه"):
            text = text.replace("انا محتاجه", "").strip()

        action = understand_command(text)

        if action == "unknown":
            speak("⚠ مش فاهم الأمر ده، جرب تاني!")
            continue

        # طلب تأكيد
        speak(f"هل تقصد {action}؟ قل نعم أو لا")
        confirm = listen_and_recognize()
        if "نعم" in confirm or "اه" in confirm:
            speak(f"تمام، هبحث عن {action}")
        elif "لا" in confirm:
            speak("طيب قول الأمر تاني")
            continue
        else:
            speak("⚠ مسمعتش إجابة واضحة، جرب تاني!")
            continue

        target_coords = None
        face_coords = None

        # معالجة الفيديو
        while True:
            ret, frame = cap.read()
            if not ret:
                speak("⚠ حصل خطأ مع الكاميرا، بحاول افتحها من جديد...")
                cap.release()
                cap = open_camera()
                if cap is None:
                    speak("⚠ فشل فتح الكاميرا، بوقف البرنامج.")
                    raise SystemExit
                continue

            # YOLO detection
            results = model(frame)
            for result in results:
                for box in result.boxes:
                    cls = model.names[int(box.cls)]
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, cls, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

                    if action == "cup" and cls == "cup":
                        target_coords = (x1, y1, x2, y2)
                    elif action == "phone" and cls == "cell phone":
                        target_coords = (x1, y1, x2, y2)

            # Face detection
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5)
            for (x, y, w, h) in faces:
                cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
                if action == "drink":
                    face_coords = (x, y, x+w, y+h)

            # تنفيذ الأمر لو لقينا الهدف
            if target_coords or face_coords or action == "hangup":
                control_robot_arm(action, target_coords, face_coords)
                break

            cv2.imshow("Camera Feed", frame)
            if cv2.waitKey(30) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()